# 21. 프로덕션 패턴 실습

**MCP 서버 · Redis 8 고급 기능 · Prometheus/Grafana 통합**

이 노트북은 v0.2에서 추가된 신규 기능들을 직접 사용해보는 실습입니다.

| 노트북 | 주제 |
|--------|------|
| 08 | Circuit Breaker 패턴 |
| 09 | Backpressure · 동시성 제어 |
| **21** | **MCP 서버 실습 + Redis 8 고급 + Prometheus** |

## 사전 요구 사항

```bash
docker compose up -d          # 전체 인프라 기동
uv run uvicorn main:app &     # FastAPI 서버 (포트 8000)
```

## 학습 목표

1. **MCP 도구**를 Python 코드에서 직접 호출하는 방법
2. **Redis 8 Bloom Filter**로 대용량 중복 필터링
3. **Redis 8 TimeSeries**로 시계열 메트릭 저장/분석
4. **Redis 8 Vector Set**으로 시맨틱 검색
5. **Prometheus 메트릭**을 Python으로 직접 쿼리
6. **Consumer Lag**를 실시간 모니터링하는 방법

In [ ]:
import asyncio
import json
import time
import httpx
import sys
sys.path.insert(0, '..')

BASE = 'http://localhost:8000'

async def api(method: str, path: str, **kwargs):
    async with httpx.AsyncClient(base_url=BASE, timeout=10) as c:
        resp = await getattr(c, method)(path, **kwargs)
        return resp.json()

# 서버 연결 확인
health = await api('get', '/health')
print('서버 상태:', json.dumps(health, indent=2, ensure_ascii=False))

---

## 섹션 1: MCP 도구 직접 호출

MCP(Model Context Protocol) 서버는 Claude와 같은 AI가 브로커를 도구로 사용할 수 있게 합니다.
여기서는 Python에서 MCP 도구를 직접 임포트해 사용해봅니다.

```
AI (Claude)  ─SSE─▶  /mcp/sse  ─▶  MCP 도구  ─▶  Redis/Kafka/RabbitMQ
Python 코드  ─직접─▶  mcp.server  ─▶  MCP 도구  ─▶  (동일)
```

In [ ]:
# MCP 도구 직접 임포트
from app.brokers import redis_broker, kafka_broker, rabbitmq_broker
from app.mcp.server import (
    memory_set, memory_get, memory_list,
    rate_limit_check, broker_health, run_benchmark,
)

# 브로커 연결
await redis_broker.connect()
print('Redis 연결 OK')

In [ ]:
# ─── 1-1. AI 컨텍스트 메모리 저장/조회 ───
# beanllm의 대화 히스토리나 중간 결과를 Redis에 저장

await memory_set('session:demo:context', json.dumps({
    'user': 'leebeanbin',
    'intent': '메시지 브로커 학습',
    'progress': 'notebook-21',
}), ttl_seconds=3600)

result = await memory_get('session:demo:context')
print('저장된 컨텍스트:', result)

keys = await memory_list('session:*')
print('세션 키 목록:', keys)

In [ ]:
# ─── 1-2. Rate Limit 체크 ───
# beanllm이 AI API 호출 전 쿼터를 확인하는 패턴

user_id = 'user:leebeanbin'

for i in range(5):
    result = json.loads(await rate_limit_check(user_id, max_requests=3, window_seconds=60))
    status = '✅ 허용' if result['allowed'] else '🚫 차단'
    print(f'요청 {i+1}: {status} | 현재 {result["current_requests"]}/3 | 남은 {result["remaining"]}')

In [ ]:
# ─── 1-3. 브로커 전체 상태 + Circuit Breaker ───
health_json = await broker_health()
health = json.loads(health_json)

print('━━━ 브로커 상태 ━━━')
for broker, info in health.items():
    connected = '🟢' if info.get('connected') else '🔴'
    cb_state = info.get('circuit_breaker', 'N/A')
    print(f'  {connected} {broker}: 연결={info["connected"]} | CB={cb_state}')

In [ ]:
# ─── 1-4. AI가 브로커 성능 비교 후 최적 선택 ───
# AI Agent 패턴: 먼저 벤치마크 후 처리량 최고 브로커 선택

results = {}
for broker in ['redis']:  # 빠른 실습을 위해 redis만 (rabbitmq/kafka는 도커 필요)
    res_json = await run_benchmark(broker, message_count=200)
    res = json.loads(res_json)
    if 'throughput_msg_per_sec' in res:
        results[broker] = res
        print(f'{broker}: {res["throughput_msg_per_sec"]:.0f} msg/s | P99={res["p99_latency_ms"]:.2f}ms')

if results:
    best = max(results, key=lambda b: results[b]['throughput_msg_per_sec'])
    print(f'\n🏆 AI 추천 브로커: {best} ({results[best]["throughput_msg_per_sec"]:.0f} msg/s)')

---

## 섹션 2: Redis 8 고급 기능

Redis 8은 Bloom Filter, TimeSeries, Vector Set을 표준 Redis 명령으로 지원합니다.

| 기능 | 구현 방식 | 사용 사례 |
|------|---------|----------|
| Bloom Filter | BITFIELD (SHA1+MD5 해싱) | 중복 URL/이메일 필터 |
| TimeSeries | Sorted Set (score=타임스탬프) | 메트릭 저장, 이상 감지 |
| Vector Set | VADD/VSIM (Redis 8.0+) | 시맨틱 검색, RAG |

In [ ]:
# ─── 2-1. Bloom Filter: 중복 URL 필터 ───
# 크롤러 시나리오: 100만 개 URL 중 이미 방문한 URL 빠르게 걸러내기

import hashlib

BF_KEY = 'demo:crawler:visited'

# 1000개 URL 등록
print('Bloom Filter에 1,000개 URL 등록 중...')
for i in range(1000):
    await api('post', '/redis/bloom/add', json={'filter_key': BF_KEY, 'item': f'https://example.com/page/{i}'})

# 필터 정보 확인
info = await api('get', f'/redis/bloom/info/{BF_KEY}')
print(f'\n필터 크기: {info["bit_size"]:,} bits ({info["bit_size"]//8//1024:.1f} KB)')
print(f'등록 항목 수: ~{info.get("estimated_items", 1000):,}개')

In [ ]:
# 방문 여부 확인
test_cases = [
    ('https://example.com/page/42',   True,  '등록된 URL'),
    ('https://example.com/page/999',  True,  '등록된 URL'),
    ('https://example.com/page/1001', False, '미등록 URL'),
    ('https://example.com/page/9999', False, '미등록 URL'),
]

print('URL 중복 체크:')
for url, expected, label in test_cases:
    resp = await api('post', '/redis/bloom/exists', json={'filter_key': BF_KEY, 'item': url})
    exists = resp['exists']
    icon = '✅' if exists == expected else '⚠️ FP'
    print(f'  {icon} [{label}] {url[-30:]} → {"이미 방문" if exists else "새 URL"}')

# 메모리 절약 효과
import sys
set_mem = sys.getsizeof(set(f'https://example.com/page/{i}' for i in range(1000)))
bf_mem = info['bit_size'] // 8
print(f'\n메모리 비교: Python set={set_mem:,}B vs Bloom Filter={bf_mem:,}B ({set_mem/bf_mem:.1f}x 절약)')

In [ ]:
# ─── 2-2. TimeSeries: 메트릭 저장 + 이상 감지 ───
import random

TS_KEY = 'demo:cpu:host1'

# 60초치 CPU 데이터 시뮬레이션 (이상값 포함)
print('CPU 메트릭 60포인트 저장 중...')
for i in range(60):
    value = 30 + random.gauss(0, 5)  # 정상 30% ±5
    if 25 <= i <= 30:  # 25~30초 구간 스파이크
        value = 85 + random.gauss(0, 3)
    value = max(0, min(100, value))
    await api('post', '/redis/ts/add', json={'series_key': TS_KEY, 'value': round(value, 2)})

print('저장 완료!')

In [ ]:
# 이상 감지: 평균 + 2σ 초과 포인트 찾기
import matplotlib.pyplot as plt
import numpy as np

data = await api('get', f'/redis/ts/range/{TS_KEY}', params={'start_ms': 0, 'end_ms': 9999999999999})
points = data['points']
values = [p['value'] for p in points]

mean, std = np.mean(values), np.std(values)
threshold = mean + 2 * std
anomalies = [(i, v) for i, v in enumerate(values) if v > threshold]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(values, label='CPU %', color='steelblue', linewidth=1.5)
ax.axhline(threshold, color='red', linestyle='--', label=f'임계값 (μ+2σ={threshold:.1f}%)')
if anomalies:
    xs, ys = zip(*anomalies)
    ax.scatter(xs, ys, color='red', zorder=5, label=f'이상 감지 {len(anomalies)}건')
ax.set_title(f'CPU 사용률 — {TS_KEY} (이상 {len(anomalies)}건 감지)')
ax.set_xlabel('시간 (샘플)')
ax.set_ylabel('CPU %')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n통계: 평균={mean:.1f}% | 표준편차={std:.1f}% | 임계값={threshold:.1f}%')
print(f'이상 감지: {len(anomalies)}건')

In [ ]:
# 최근 값 조회 + 오래된 데이터 정리
latest = await api('get', f'/redis/ts/latest/{TS_KEY}')
print(f'최근 값: {latest["value"]}% (at {latest["timestamp_ms"]}ms)')

# 오래된 데이터 삭제 (TTL이 없는 SortedSet 기반 TS의 수동 정리)
cutoff_ms = int(time.time() * 1000) - 30_000  # 30초 이전
trim = await api('delete', f'/redis/ts/trim/{TS_KEY}', params={'older_than_ms': cutoff_ms})
print(f'{trim["removed"]}개 포인트 삭제 (30초 이전 데이터)')

In [ ]:
# ─── 2-3. Vector Set: 시맨틱 검색 ───
# Redis 8.0+ VADD/VSIM 필요. 지원하지 않으면 beanllm 시나리오 설명으로 대체.

VS_KEY = 'demo:docs'

docs = [
    ('doc:kafka',     [0.9, 0.1, 0.1, 0.05], 'Kafka는 고처리량 분산 이벤트 스트리밍 플랫폼입니다.'),
    ('doc:rabbitmq',  [0.1, 0.9, 0.1, 0.05], 'RabbitMQ는 AMQP 기반 메시지 큐 브로커입니다.'),
    ('doc:redis',     [0.1, 0.1, 0.9, 0.05], 'Redis는 인메모리 데이터 구조 서버입니다.'),
    ('doc:beanllm',   [0.3, 0.3, 0.3, 0.9],  'beanllm은 세 브로커를 내부적으로 모두 활용하는 AI 프레임워크입니다.'),
]

# 벡터 등록
for doc_id, vec, text in docs:
    resp = await api('post', '/redis/vector/add', json={'vset_key': VS_KEY, 'element_id': doc_id, 'vector': vec})
    if 'error' in resp:
        print(f'⚠️  Vector Set 미지원 (Redis 8.0+ 필요): {resp["error"]}')
        print('   beanllm 내부 RAGChain은 이 패턴으로 시맨틱 캐시를 구현합니다.')
        break
    print(f'  벡터 등록: {doc_id}')
else:
    # 쿼리: "이벤트 스트리밍"과 유사한 문서 찾기
    query_vec = [0.85, 0.05, 0.05, 0.1]  # Kafka에 가까운 벡터
    results = await api('post', '/redis/vector/search', json={
        'vset_key': VS_KEY,
        'query_vector': query_vec,
        'top_k': 3,
    })
    print('\n시맨틱 검색 결과 (쿼리: 이벤트 스트리밍과 유사한 문서):')
    for r in results.get('results', []):
        text = next((t for d, _, t in docs if d == r['id']), '')
        print(f'  [{r["id"]}] 유사도={r.get("score", "N/A")} → {text[:50]}')

---

## 섹션 3: Prometheus 메트릭 직접 쿼리

FastAPI가 `/metrics` 엔드포인트로 Prometheus 형식의 메트릭을 노출합니다.
Python에서 직접 쿼리해 현재 상태를 모니터링할 수 있습니다.

```
FastAPI → /metrics → Prometheus(9090) → Grafana(3000)
                  ↗
        이 셀에서 직접 쿼리
```

In [ ]:
# ─── 3-1. /metrics 엔드포인트 직접 파싱 ───

async with httpx.AsyncClient(base_url=BASE) as c:
    resp = await c.get('/metrics')
    metrics_text = resp.text

# 관심 메트릭 추출
interesting = ['circuit_breaker_state', 'backpressure_active', 'kafka_consumer_group_lag', 'broker_publish']

print('━━━ 현재 메트릭 스냅샷 ━━━')
for line in metrics_text.split('\n'):
    if line.startswith('#'):
        continue
    if any(key in line for key in interesting):
        print(' ', line)

In [ ]:
# ─── 3-2. Consumer Lag API 엔드포인트 ───

lag_data = await api('get', '/monitoring/kafka-lag')
print('Consumer Lag 스냅샷:')
print(f'  모니터링 그룹: {len(lag_data["monitored_groups"])}개')

if lag_data['lag_data']:
    for entry in lag_data['lag_data']:
        status = '✅ 정상' if entry['total'] < 1000 else ('⚠️ 주의' if entry['total'] < 10000 else '🚨 알람')
        print(f'  {status} {entry["group"]}/{entry["topic"]}: lag={entry["total"]}')
else:
    print('  (Kafka 미연결 or 아직 측정 전 — 15초 후 재시도)')

In [ ]:
# ─── 3-3. Circuit Breaker 상태 관찰 ───

cb_status = await api('get', '/resilience/circuit-breakers')

print('Circuit Breaker 상태:')
state_icon = {'closed': '🟢', 'open': '🔴', 'half_open': '🟡'}
for broker, info in cb_status.items():
    icon = state_icon.get(info['state'], '⚪')
    print(f'  {icon} {broker}: {info["state"]} | 실패={info["failure_count"]}/{info["threshold"]} | '
          f'retry_after={info["retry_after_seconds"]}s')

In [ ]:
# ─── 3-4. Backpressure 상태 ───

bp_status = await api('get', '/resilience/backpressure')

print('Backpressure 가드 상태:')
for broker, info in bp_status.items():
    load_pct = (info['active'] / info['max_concurrent']) * 100 if info['max_concurrent'] > 0 else 0
    bar = '█' * int(load_pct / 10) + '░' * (10 - int(load_pct / 10))
    overloaded = ' 🚨 OVERLOADED' if info['overloaded'] else ''
    print(f'  {broker}: [{bar}] {load_pct:.0f}% ({info["active"]}/{info["max_concurrent"]}){overloaded}')

---

## 섹션 4: 프로덕션 체크리스트

### 브로커 선택 가이드

| 상황 | 추천 브로커 | 이유 |
|------|------------|------|
| LLM 응답 스트리밍 | Redis Stream | 낮은 레이턴시, XADD/XREAD |
| AI 에이전트 작업 분배 | RabbitMQ Work Queue | 안정적 전달, Priority, DLQ |
| 대용량 이벤트 로그 | Kafka | 높은 처리량, 영구 저장 |
| 시맨틱 캐시 | Redis Vector Set | VSIM 코사인 유사도 |
| Rate Limiting | Redis ZADD | 슬라이딩 윈도우, O(log N) |

### beanllm 내부 매핑

```python
RAGChain.query()    →  Redis VSIM (Vector Set)
rate_limit()        →  Redis ZADD (슬라이딩 윈도우)
Agent.run_parallel()→  RabbitMQ Work Queue
stream_chat()       →  Redis Stream XADD/XREAD
audit_log()         →  Kafka (순서 보장, 영구 저장)
```

### v0.2 신규 기능 요약

| 기능 | 엔드포인트 | 설명 |
|------|-----------|------|
| MCP 서버 | `/mcp/sse` | AI가 브로커를 직접 도구로 사용 |
| Circuit Breaker | `/resilience/circuit-breakers` | 장애 격리 |
| Backpressure | `/resilience/backpressure` | 동시성 제어 |
| Consumer Lag | `/monitoring/kafka-lag` | Kafka 소비 지연 모니터링 |
| Bloom Filter | `/redis/bloom/*` | 확률적 중복 필터 |
| TimeSeries | `/redis/ts/*` | 시계열 저장소 |
| Vector Set | `/redis/vector/*` | 시맨틱 검색 (Redis 8.0+) |

In [ ]:
# ─── 4-1. Grafana 대시보드 접속 안내 ───

print('📊 Grafana 대시보드 확인')
print('=' * 50)
print('URL:      http://localhost:3000')
print('계정:     admin / admin')
print('대시보드:  Message Broker Lab (자동 로드됨)')
print()
print('대시보드 패널:')
print('  1. Publish Latency P50/P99  ← 브로커 성능 비교')
print('  2. 처리량 (msg/s)            ← 초당 메시지 처리 수')
print('  3. Kafka Consumer Lag       ← 소비 지연 (0이 이상적)')
print('  4. Circuit Breaker 상태     ← CLOSED/OPEN/HALF_OPEN')
print('  5. Backpressure 활성 요청   ← 동시성 사용률')
print()
print('PromQL 예시:')
print('  Consumer Lag: kafka_consumer_group_lag_sum')
print('  CB 상태:      circuit_breaker_state{broker="redis"}')
print('  P99 레이턴시: histogram_quantile(0.99, rate(broker_publish_latency_seconds_bucket[1m]))')

In [ ]:
# ─── 4-2. MCP 연결 방법 요약 ───

print('🤖 MCP 서버 연결 방법')
print('=' * 50)
print()
print('[방법 1] Claude Code (SSE — 서버 실행 중 필요)')
print('.claude/settings.json:')
print(json.dumps({'mcpServers': {'broker-lab': {'type': 'sse', 'url': 'http://localhost:8000/mcp/sse'}}}, indent=2))
print()
print('[방법 2] Claude Desktop (stdio)')
print('~/Library/Application Support/Claude/claude_desktop_config.json:')
print(json.dumps({'mcpServers': {'broker-lab': {'command': 'uv', 'args': ['run', 'mcp_server.py']}}}, indent=2))
print()
print('사용 가능한 MCP 도구 (15개):')
tools = [
    ('메모리', 'memory_set / memory_get / memory_delete / memory_list'),
    ('벡터',   'vector_store / vector_search'),
    ('스트림', 'stream_publish / stream_read'),
    ('이벤트', 'kafka_emit / kafka_topic_info'),
    ('태스크', 'task_enqueue / task_queue_info'),
    ('제한',   'rate_limit_check'),
    ('모니터', 'broker_health / run_benchmark'),
]
for category, tool_list in tools:
    print(f'  [{category}] {tool_list}')